# Preprocessing & Feature Engineering
Cleaning, merging, creating temporal features, lag features, and preparing the dataset for modeling.

**Input:** Raw CSVs  
**Output:** `data/metro_processed.csv` — ready for model training

In [1]:
import pandas as pd
import numpy as np

## Load & Merge Datasets

In [2]:
df_board = pd.read_csv('data/station-hourly.csv', sep=';')
df_exit = pd.read_csv('data/station-hourly-exits.csv', sep=';')

In [3]:
df_board.rename(columns={'Ridership': 'Boarding_Count'}, inplace=True)
df_exit.rename(columns={'Ridership': 'Exit_Count'}, inplace=True)

In [4]:
df = pd.merge(df_board, df_exit, on=['Date', 'Hour', 'Station'], how='outer')
df.shape

(95616, 5)

In [5]:
df.isnull().sum()

Date                 0
Hour                 0
Station              0
Boarding_Count    3336
Exit_Count           0
dtype: int64

In [6]:
# fill NaNs — these are from Yellow Line stations where boarding wasnt logged on some dates
df['Boarding_Count'] = df['Boarding_Count'].fillna(0).astype(int)
df['Exit_Count'] = df['Exit_Count'].fillna(0).astype(int)

In [7]:
df.isnull().sum()

Date              0
Hour              0
Station           0
Boarding_Count    0
Exit_Count        0
dtype: int64

In [8]:
# create proper datetime
df['DateTime'] = pd.to_datetime(df['Date'] + ' ' + df['Hour'].astype(str).str.zfill(2) + ':00:00')
df = df.sort_values(['Station', 'DateTime']).reset_index(drop=True)

In [9]:
df.head()

,Date,Hour,Station,Boarding_Count,Exit_Count,DateTime
0,2025-08-01,0,Attiguppe,0,0,2025-08-01 00:00:00
1,2025-08-01,1,Attiguppe,0,0,2025-08-01 01:00:00
2,2025-08-01,2,Attiguppe,0,0,2025-08-01 02:00:00
3,2025-08-01,3,Attiguppe,0,0,2025-08-01 03:00:00
4,2025-08-01,4,Attiguppe,6,0,2025-08-01 04:00:00


In [10]:
df.shape

(95616, 6)

## Outlier Capping

In [11]:
cap_99 = df['Boarding_Count'].quantile(0.99)
print(f"99th percentile: {cap_99}")

99th percentile: 2352.0


In [12]:
# not removing outliers — they are real peak surges. just capping extreme values
df['Boarding_Capped'] = df['Boarding_Count'].clip(upper=cap_99)

In [13]:
df['Boarding_Capped'].describe()

count    95616.000000
mean       348.303516
std        459.188799
min          0.000000
25%          5.000000
50%        196.000000
75%        482.000000
max       2352.000000
Name: Boarding_Capped, dtype: float64

## Temporal Features

In [14]:
df['DayOfWeek'] = df['DateTime'].dt.dayofweek
df['DayOfMonth'] = df['DateTime'].dt.day
df['WeekOfYear'] = df['DateTime'].dt.isocalendar().week.astype(int)

In [15]:
df['Is_Weekend'] = (df['DayOfWeek'] >= 5).astype(int)

In [16]:
df.head()

,Date,Hour,Station,Boarding_Count,Exit_Count,DateTime,Boarding_Capped,DayOfWeek,DayOfMonth,WeekOfYear,Is_Weekend
0,2025-08-01,0,Attiguppe,0,0,2025-08-01 00:00:00,0,4,1,31,0
1,2025-08-01,1,Attiguppe,0,0,2025-08-01 01:00:00,0,4,1,31,0
2,2025-08-01,2,Attiguppe,0,0,2025-08-01 02:00:00,0,4,1,31,0
3,2025-08-01,3,Attiguppe,0,0,2025-08-01 03:00:00,0,4,1,31,0
4,2025-08-01,4,Attiguppe,6,0,2025-08-01 04:00:00,6,4,1,31,0


## Cyclic Encoding
So the model knows hour 23 and hour 0 are adjacent, not far apart.

In [17]:
df['Hour_Sin'] = np.sin(2 * np.pi * df['Hour'] / 24)
df['Hour_Cos'] = np.cos(2 * np.pi * df['Hour'] / 24)

In [18]:
df['Day_Sin'] = np.sin(2 * np.pi * df['DayOfWeek'] / 7)
df['Day_Cos'] = np.cos(2 * np.pi * df['DayOfWeek'] / 7)

In [19]:
df.head()

,Date,Hour,Station,Boarding_Count,Exit_Count,DateTime,Boarding_Capped,DayOfWeek,DayOfMonth,WeekOfYear,Is_Weekend,Hour_Sin,Hour_Cos,Day_Sin,Day_Cos
0,2025-08-01,0,Attiguppe,0,0,2025-08-01 00:00:00,0,4,1,31,0,0.000000,1.000000,-0.433884,-0.900969
1,2025-08-01,1,Attiguppe,0,0,2025-08-01 01:00:00,0,4,1,31,0,0.258819,0.965926,-0.433884,-0.900969
2,2025-08-01,2,Attiguppe,0,0,2025-08-01 02:00:00,0,4,1,31,0,0.500000,0.866025,-0.433884,-0.900969
3,2025-08-01,3,Attiguppe,0,0,2025-08-01 03:00:00,0,4,1,31,0,0.707107,0.707107,-0.433884,-0.900969
4,2025-08-01,4,Attiguppe,6,0,2025-08-01 04:00:00,6,4,1,31,0,0.866025,0.500000,-0.433884,-0.900969


## Peak Hour Flags

In [20]:
# morning rush 7-10am, evening rush 5-8pm on weekdays only
df['Is_Morning_Peak'] = ((df['Hour'] >= 7) & (df['Hour'] <= 10) & (df['Is_Weekend'] == 0)).astype(int)
df['Is_Evening_Peak'] = ((df['Hour'] >= 17) & (df['Hour'] <= 20) & (df['Is_Weekend'] == 0)).astype(int)
df['Is_Peak_Hour'] = (df['Is_Morning_Peak'] | df['Is_Evening_Peak']).astype(int)

In [21]:
df['Is_Peak_Hour'].value_counts()

Is_Peak_Hour
0    73040
1    22576
Name: count, dtype: int64

In [22]:
# sanity check — peak should have higher traffic
df.groupby('Is_Peak_Hour')['Boarding_Count'].mean()

Is_Peak_Hour
0    242.224658
1    715.175097
Name: Boarding_Count, dtype: float64

## Calendar Feature Decision: Holiday Flag
We considered adding an `Is_Holiday` binary flag for Indian public holidays (Independence Day, Janmashtami, etc.).
However, with only 2 months of data containing 5-6 holiday dates, the model cannot learn a statistically reliable holiday traffic pattern from such a small sample.

**Decision:** We rely on `Is_Weekend` and `DayOfWeek` to capture non-working day patterns, and lag features (`Lag_1h`, `Lag_24h`) to reflect actual real-time traffic conditions on any given day.

In [23]:
df.head()

,Date,Hour,Station,Boarding_Count,Exit_Count,DateTime,Boarding_Capped,DayOfWeek,DayOfMonth,WeekOfYear,Is_Weekend,Hour_Sin,Hour_Cos,Day_Sin,Day_Cos,Is_Morning_Peak,Is_Evening_Peak,Is_Peak_Hour
0,2025-08-01,0,Attiguppe,0,0,2025-08-01 00:00:00,0,4,1,31,0,0.000000,1.000000,-0.433884,-0.900969,0,0,0
1,2025-08-01,1,Attiguppe,0,0,2025-08-01 01:00:00,0,4,1,31,0,0.258819,0.965926,-0.433884,-0.900969,0,0,0
2,2025-08-01,2,Attiguppe,0,0,2025-08-01 02:00:00,0,4,1,31,0,0.500000,0.866025,-0.433884,-0.900969,0,0,0
3,2025-08-01,3,Attiguppe,0,0,2025-08-01 03:00:00,0,4,1,31,0,0.707107,0.707107,-0.433884,-0.900969,0,0,0
4,2025-08-01,4,Attiguppe,6,0,2025-08-01 04:00:00,6,4,1,31,0,0.866025,0.500000,-0.433884,-0.900969,0,0,0


## Lag Features
Each lag is computed **per station** so we dont leak data across stations.

In [24]:
df = df.sort_values(['Station', 'DateTime']).reset_index(drop=True)

In [25]:
df['Lag_1h'] = df.groupby('Station')['Boarding_Count'].shift(1)
df['Lag_2h'] = df.groupby('Station')['Boarding_Count'].shift(2)
df['Lag_24h'] = df.groupby('Station')['Boarding_Count'].shift(24)  # same hour yesterday

In [26]:
df[['Station', 'DateTime', 'Boarding_Count', 'Lag_1h', 'Lag_2h', 'Lag_24h']].head(30)

,Station,DateTime,Boarding_Count,Lag_1h,Lag_2h,Lag_24h
0,Attiguppe,2025-08-01 00:00:00,0,NaN,NaN,NaN
1,Attiguppe,2025-08-01 01:00:00,0,0.0,NaN,NaN
2,Attiguppe,2025-08-01 02:00:00,0,0.0,0.0,NaN
3,Attiguppe,2025-08-01 03:00:00,0,0.0,0.0,NaN
4,Attiguppe,2025-08-01 04:00:00,6,0.0,0.0,NaN
5,Attiguppe,2025-08-01 05:00:00,65,6.0,0.0,NaN
6,Attiguppe,2025-08-01 06:00:00,219,65.0,6.0,NaN
7,Attiguppe,2025-08-01 07:00:00,646,219.0,65.0,NaN
8,Attiguppe,2025-08-01 08:00:00,1338,646.0,219.0,NaN
9,Attiguppe,2025-08-01 09:00:00,1818,1338.0,646.0,NaN


## Rolling Window Features

In [27]:
df['Rolling_3h'] = df.groupby('Station')['Boarding_Count'].transform(
    lambda x: x.shift(1).rolling(window=3, min_periods=1).mean()
)

In [28]:
df['Rolling_3h_Std'] = df.groupby('Station')['Boarding_Count'].transform(
    lambda x: x.shift(1).rolling(window=3, min_periods=1).std()
)

In [29]:
df.head()

,Date,Hour,Station,Boarding_Count,Exit_Count,DateTime,Boarding_Capped,DayOfWeek,DayOfMonth,WeekOfYear,...,Day_Sin,Day_Cos,Is_Morning_Peak,Is_Evening_Peak,Is_Peak_Hour,Lag_1h,Lag_2h,Lag_24h,Rolling_3h,Rolling_3h_Std
0,2025-08-01,0,Attiguppe,0,0,2025-08-01 00:00:00,0,4,1,31,...,-0.433884,-0.900969,0,0,0,NaN,NaN,NaN,NaN,NaN
1,2025-08-01,1,Attiguppe,0,0,2025-08-01 01:00:00,0,4,1,31,...,-0.433884,-0.900969,0,0,0,0.0,NaN,NaN,0.0,NaN
2,2025-08-01,2,Attiguppe,0,0,2025-08-01 02:00:00,0,4,1,31,...,-0.433884,-0.900969,0,0,0,0.0,0.0,NaN,0.0,0.0
3,2025-08-01,3,Attiguppe,0,0,2025-08-01 03:00:00,0,4,1,31,...,-0.433884,-0.900969,0,0,0,0.0,0.0,NaN,0.0,0.0
4,2025-08-01,4,Attiguppe,6,0,2025-08-01 04:00:00,6,4,1,31,...,-0.433884,-0.900969,0,0,0,0.0,0.0,NaN,0.0,0.0


## Station Encoding

In [30]:
# using mean boarding as a proxy for station "busyness"
station_avg = df.groupby('Station')['Boarding_Count'].mean()
station_avg.sort_values(ascending=False).head(10)

Station
Nadaprabhu Kempegowda Station, Majestic    1431.883681
Benniganahalli                             1085.638889
Indiranagar                                 910.534722
Mahatma Gandhi Road                         874.284722
Krishnarajapura                             785.163194
Mantri Square Sampige Road                  691.423611
Chickpete                                   640.500000
Cubbon Park                                 578.330729
Yeshwantpur                                 578.289062
Baiyappanahalli                             559.569444
Name: Boarding_Count, dtype: float64

In [31]:
df['Station_AvgTraffic'] = df['Station'].map(station_avg)

In [32]:
df.head()

,Date,Hour,Station,Boarding_Count,Exit_Count,DateTime,Boarding_Capped,DayOfWeek,DayOfMonth,WeekOfYear,...,Day_Cos,Is_Morning_Peak,Is_Evening_Peak,Is_Peak_Hour,Lag_1h,Lag_2h,Lag_24h,Rolling_3h,Rolling_3h_Std,Station_AvgTraffic
0,2025-08-01,0,Attiguppe,0,0,2025-08-01 00:00:00,0,4,1,31,...,-0.900969,0,0,0,NaN,NaN,NaN,NaN,NaN,350.94184
1,2025-08-01,1,Attiguppe,0,0,2025-08-01 01:00:00,0,4,1,31,...,-0.900969,0,0,0,0.0,NaN,NaN,0.0,NaN,350.94184
2,2025-08-01,2,Attiguppe,0,0,2025-08-01 02:00:00,0,4,1,31,...,-0.900969,0,0,0,0.0,0.0,NaN,0.0,0.0,350.94184
3,2025-08-01,3,Attiguppe,0,0,2025-08-01 03:00:00,0,4,1,31,...,-0.900969,0,0,0,0.0,0.0,NaN,0.0,0.0,350.94184
4,2025-08-01,4,Attiguppe,6,0,2025-08-01 04:00:00,6,4,1,31,...,-0.900969,0,0,0,0.0,0.0,NaN,0.0,0.0,350.94184


## Handle NaN from Lag Features

In [33]:
df.isnull().sum()

Date                     0
Hour                     0
Station                  0
Boarding_Count           0
Exit_Count               0
DateTime                 0
Boarding_Capped          0
DayOfWeek                0
DayOfMonth               0
WeekOfYear               0
Is_Weekend               0
Hour_Sin                 0
Hour_Cos                 0
Day_Sin                  0
Day_Cos                  0
Is_Morning_Peak          0
Is_Evening_Peak          0
Is_Peak_Hour             0
Lag_1h                  83
Lag_2h                 166
Lag_24h               1992
Rolling_3h              83
Rolling_3h_Std         166
Station_AvgTraffic       0
dtype: int64

In [34]:
# NaNs are from the first 24 hours of each station — no historical data available for those
df_clean = df.dropna().copy()
print(f"Before: {df.shape[0]}")
print(f"After:  {df_clean.shape[0]}")
print(f"Dropped: {df.shape[0] - df_clean.shape[0]}")

Before: 95616
After:  93624
Dropped: 1992


In [35]:
df_clean.isnull().sum()

Date                  0
Hour                  0
Station               0
Boarding_Count        0
Exit_Count            0
DateTime              0
Boarding_Capped       0
DayOfWeek             0
DayOfMonth            0
WeekOfYear            0
Is_Weekend            0
Hour_Sin              0
Hour_Cos              0
Day_Sin               0
Day_Cos               0
Is_Morning_Peak       0
Is_Evening_Peak       0
Is_Peak_Hour          0
Lag_1h                0
Lag_2h                0
Lag_24h               0
Rolling_3h            0
Rolling_3h_Std        0
Station_AvgTraffic    0
dtype: int64

In [36]:
df_clean.head()

,Date,Hour,Station,Boarding_Count,Exit_Count,DateTime,Boarding_Capped,DayOfWeek,DayOfMonth,WeekOfYear,...,Day_Cos,Is_Morning_Peak,Is_Evening_Peak,Is_Peak_Hour,Lag_1h,Lag_2h,Lag_24h,Rolling_3h,Rolling_3h_Std,Station_AvgTraffic
24,2025-08-02,0,Attiguppe,0,0,2025-08-02 00:00:00,0,5,2,31,...,-0.222521,0,0,0,23.0,141.0,0.0,122.000000,91.000000,350.94184
25,2025-08-02,1,Attiguppe,0,0,2025-08-02 01:00:00,0,5,2,31,...,-0.222521,0,0,0,0.0,23.0,0.0,54.666667,75.646106,350.94184
26,2025-08-02,2,Attiguppe,0,0,2025-08-02 02:00:00,0,5,2,31,...,-0.222521,0,0,0,0.0,0.0,0.0,7.666667,13.279056,350.94184
27,2025-08-02,3,Attiguppe,0,0,2025-08-02 03:00:00,0,5,2,31,...,-0.222521,0,0,0,0.0,0.0,0.0,0.000000,0.000000,350.94184
28,2025-08-02,4,Attiguppe,10,0,2025-08-02 04:00:00,10,5,2,31,...,-0.222521,0,0,0,0.0,0.0,6.0,0.000000,0.000000,350.94184


In [37]:
df_clean.shape

(93624, 24)

In [38]:
df_clean.dtypes

Date                          object
Hour                           int64
Station                       object
Boarding_Count                 int32
Exit_Count                     int32
DateTime              datetime64[ns]
Boarding_Capped                int32
DayOfWeek                      int32
DayOfMonth                     int32
WeekOfYear                     int32
Is_Weekend                     int32
Hour_Sin                     float64
Hour_Cos                     float64
Day_Sin                      float64
Day_Cos                      float64
Is_Morning_Peak                int32
Is_Evening_Peak                int32
Is_Peak_Hour                   int32
Lag_1h                       float64
Lag_2h                       float64
Lag_24h                      float64
Rolling_3h                   float64
Rolling_3h_Std               float64
Station_AvgTraffic           float64
dtype: object

## Save Processed Dataset

In [39]:
df_clean.to_csv('data/metro_processed.csv', index=False)
print(f"Saved {df_clean.shape[0]} rows to data/metro_processed.csv")

Saved 93624 rows to data/metro_processed.csv


In [40]:
df_clean.shape

(93624, 24)

**Features created:**
- Temporal: DayOfWeek, DayOfMonth, WeekOfYear, Is_Weekend
- Cyclic: Hour_Sin, Hour_Cos, Day_Sin, Day_Cos
- Peak: Is_Morning_Peak, Is_Evening_Peak, Is_Peak_Hour
- Lag: Lag_1h, Lag_2h, Lag_24h
- Rolling: Rolling_3h, Rolling_3h_Std
- Station: Station_AvgTraffic
- Capped: Boarding_Capped (99th percentile clipping)

**Next:** Move to Model Building notebook